# 01 · Профиль данных и слой 0 проверок

Ноутбук обязан проходить сверху вниз на перезапущенном ядре.
Воспроизводит контрольные числа из `DATA_PASSPORT.md` и закрывает проверки
0.1, 0.2, 0.3, 1, 2, 7, 15, 21, 22.

## Что это за данные

Транзакционная активность клиентов банка за 2022 год, собранная для анализа
влияния кэшбэк-программы на лояльность — документация набора [dano-cashback-doc].

**Окна наблюдения разной длины, и это определяет всё чтение долей:**

| колонки | окно | длина |
|---|---|---|
| `*_before`, `cb_merch_before`, `cb_bank_before` | «с 2022-01-01 по 2022-08-01» | 7 месяцев |
| `*_last_3_month`, `cb_*_last_3_month` | «последние 3 месяца» | 3 месяца |

**Точных границ окна «последние 3 месяца» в документации нет, и получить их
не удалось.** По дате выгрузки (2022-11-15) это примерно август–ноябрь 2022, но
как предположение, а не как факт. Отсюда два запрета на всю работу:

1. никаких утверждений с привязкой к календарным месяцам и к сезонности —
   проверить их нечем;
2. возможное перекрытие окон в августе 2022 не исключено и не проверяемо, поэтому
   «до» и «после» не трактуются как строго последовательные периоды.

Доли двух окон напрямую несравнимы: часть различий создаёт длина окна, и это
считается отдельной проверкой (22) ниже.


In [13]:
# параметры и константы
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd().parent
sys.path.insert(0, str(PROJECT_DIR / "src"))

RAW = PROJECT_DIR / "data" / "raw" / "cashback_raw_2022-11-15.csv"
SEED = 42
ALPHA, POWER = 0.05, 0.80
Z_SUM = 1.959964 + 0.841621  # двусторонняя альтернатива

# колонки-нули по всей выборке, выбрасываются (DATA_PASSPORT, ИСКЛЮЧЕНИЯ)
DROP_COLS = ["Taxi", "Taxi_before"]
TARGET_SRC = "cb_merch_last_3_month"   # целевое действие: покупка у партнёра
RISK_SRC = "cb_bank_last_3_month"      # risk-метрика: выплаты банка


In [14]:
df = pd.read_csv(RAW)
print("строк:", len(df), "| колонок:", df.shape[1])
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
print("после удаления мёртвых колонок:", df.shape)


строк: 50000 | колонок: 42
после удаления мёртвых колонок: (50000, 40)


## Проверка 0.1 — единица наблюдения

In [15]:
n_rows, n_keys = len(df), df["client_id"].nunique()
print(f"строк {n_rows}, уникальных client_id {n_keys}")
assert n_rows == n_keys, "ключ клиента не уникален — дальше не считаем"
print("вердикт: сошлось")


строк 50000, уникальных client_id 50000
вердикт: сошлось


## Проверка 0.2 / 0.3 — пропуски, типы, содержательные нули

Ноль в тратах означает «не покупал», а не «неизвестно». Это фиксируется, а не чинится.


In [16]:
na = df.isna().sum()
print("колонок с пропусками:", int((na > 0).sum()))
print("нечисловых колонок:", list(df.dtypes[df.dtypes == object].index))

zero_share = (df.drop(columns=["client_id"]) == 0).mean().sort_values(ascending=False)
print("\nтоп-5 колонок по доле нулей:")
print(zero_share.head(5).round(4).to_string())


колонок с пропусками: 0
нечисловых колонок: []

топ-5 колонок по доле нулей:
Jewelry       0.8739
Cinema        0.8621
Air           0.8550
Pet shop      0.7871
Air_before    0.7832


## Проверка 7 — странные значения

In [17]:
num = df.drop(columns=["client_id"])
neg_counts = (num < 0).sum()
neg_counts = neg_counts[neg_counts > 0].sort_values(ascending=False)

# Счётчика «колонок с отрицательными значениями: 4» недостаточно: проверка,
# которая не говорит, ГДЕ отрицательные и сколько их, ничего не проверяет.
rows = []
for column, count in neg_counts.items():
    values = df.loc[df[column] < 0, column]
    rows.append({
        "колонка": column,
        "строк < 0": int(count),
        "доля": count / len(df),
        "минимум": values.min(),
        "медиана < 0": values.median(),
        "сумма < 0": values.sum(),
    })
negatives = pd.DataFrame(rows)
print("колонок с отрицательными значениями:", len(negatives))
if len(negatives):
    print(negatives.round(4).to_string(index=False))
    print("\nЭто сторно кэшбэка при возврате покупки: отрицательные значения есть")
    print("только в денежных колонках, в тратах по категориям их нет.")
    affected = ((df["cb_merch_last_3_month"] < 0) | (df["cb_bank_last_3_month"] < 0)).sum()
    print("клиентов с отрицательным кэшбэком в окне «после»: %d (%.4f)"
          % (affected, affected / len(df)))
    print("\nНа целевое действие влияет пренебрежимо: флаг cb_merch_last_3_month > 0")
    print("относит клиента со сторно к непокупавшим, таких %d (%.4f) —"
          % ((df["cb_merch_last_3_month"] < 0).sum(), (df["cb_merch_last_3_month"] < 0).mean()))
    print("базовая доля от этого не меняется в четвёртом знаке. Определение целевого")
    print("действия: покупка у партнёра = ПОЛОЖИТЕЛЬНЫЙ итоговый кэшбэк за окно.")

print("\nскошенность (среднее / медиана) по денежным колонкам:")
for c in [TARGET_SRC, RISK_SRC, "cb_merch_before", "cb_bank_before"]:
    v = df.loc[df[c] > 0, c]
    print(f"  {c:24s} среднее {v.mean():8.2f}  медиана {v.median():7.1f}  отношение {v.mean() / v.median():.2f}")

колонок с отрицательными значениями: 4
              колонка  строк < 0   доля  минимум  медиана < 0  сумма < 0
 cb_bank_last_3_month         71 0.0014  -4950.0       -493.0   -55242.0
cb_merch_last_3_month         29 0.0006 -31499.0       -304.0   -49445.0
       cb_bank_before         21 0.0004  -4702.0       -363.0   -15863.0
      cb_merch_before         12 0.0002 -15965.0       -318.0   -23718.0

Это сторно кэшбэка при возврате покупки: отрицательные значения есть
только в денежных колонках, в тратах по категориям их нет.
клиентов с отрицательным кэшбэком в окне «после»: 82 (0.0016)

На целевое действие влияет пренебрежимо: флаг cb_merch_last_3_month > 0
относит клиента со сторно к непокупавшим, таких 29 (0.0006) —
базовая доля от этого не меняется в четвёртом знаке. Определение целевого
действия: покупка у партнёра = ПОЛОЖИТЕЛЬНЫЙ итоговый кэшбэк за окно.

скошенность (среднее / медиана) по денежным колонкам:
  cb_merch_last_3_month    среднее   632.05  медиана   206.0  отношение

## Проверка 15 — образуют ли счётчики воронку

Гипотеза версии 2 плана: показ → просмотр → активация. Проверяем до того, как
строить на ней генератор.

Документация даёт второй, независимый довод против этой гипотезы:
`cnt_activation` и `cnt_view` измерены **за последние 3 месяца**, то есть в том
же окне, что целевое действие, а `cnt_communication` приведён без указания окна.
Шагами одной воронки такие величины быть не могут, и признаками, доступными до
кампании, — тоже. См. `DECISIONS.md`, решения №13 и №31.


In [18]:
bad_view = (df["cnt_view"] > df["cnt_communication"]).mean()
bad_act = (df["cnt_activation"] > df["cnt_view"]).mean()
no_comm = df[df["cnt_communication"] == 0]
print(f"cnt_view > cnt_communication: {bad_view:.4f} клиентов")
print(f"cnt_activation > cnt_view:    {bad_act:.4f} клиентов")
print(f"\nбез коммуникаций вовсе: {len(no_comm)} ({len(no_comm)/len(df):.4f})")
print(f"  из них с просмотрами:  {(no_comm['cnt_view'] > 0).mean():.4f}")
print(f"  из них с активациями:  {(no_comm['cnt_activation'] > 0).mean():.4f}")
print(f"  медиана просмотров:    {no_comm['cnt_view'].median():.0f}")
print("\nвердикт: СТОП по гипотезе воронки — счётчики описывают витрину кэшбэка,")
print("а не отклик на коммуникацию. Они переходят в ковариаты (решение №13).")


cnt_view > cnt_communication: 0.9153 клиентов
cnt_activation > cnt_view:    0.0112 клиентов

без коммуникаций вовсе: 17142 (0.3428)
  из них с просмотрами:  0.8475
  из них с активациями:  0.8468
  медиана просмотров:    124

вердикт: СТОП по гипотезе воронки — счётчики описывают витрину кэшбэка,
а не отклик на коммуникацию. Они переходят в ковариаты (решение №13).


## Проверка 21 — контрольные числа паспорта

In [19]:
y = (df[TARGET_SRC] > 0).astype(int)          # целевое действие
risk = (df[RISK_SRC] > 0).astype(int)          # risk-метрика

control = {
    "доля покупок у партнёра, после": y.mean(),
    "доля покупок у партнёра, до": (df["cb_merch_before"] > 0).mean(),
    "доля с банковским кэшбэком, после": risk.mean(),
    "доля с банковским кэшбэком, до": (df["cb_bank_before"] > 0).mean(),
    "доля получавших коммуникации": (df["cnt_communication"] > 0).mean(),
}
expected = {
    "доля покупок у партнёра, после": 0.4795,
    "доля покупок у партнёра, до": 0.6737,
    "доля с банковским кэшбэком, после": 0.3838,
    "доля с банковским кэшбэком, до": 0.6569,
    "доля получавших коммуникации": 0.6572,
}
for k, v in control.items():
    delta = abs(v - expected[k]) / expected[k]
    print(f"{k:38s} {v:.4f}  (паспорт {expected[k]:.4f}, расхождение {delta:.2%})")
    assert delta < 0.01, f"{k}: расхождение больше 1%"
print("\nвердикт: сошлось")


доля покупок у партнёра, после         0.4795  (паспорт 0.4795, расхождение 0.00%)
доля покупок у партнёра, до            0.6737  (паспорт 0.6737, расхождение 0.01%)
доля с банковским кэшбэком, после      0.3838  (паспорт 0.3838, расхождение 0.01%)
доля с банковским кэшбэком, до         0.6569  (паспорт 0.6569, расхождение 0.01%)
доля получавших коммуникации           0.6572  (паспорт 0.6572, расхождение 0.01%)

вердикт: сошлось


## Проверка 22 — объясняется ли падение долей длиной окна

Доли ненулевого кэшбэка в окне «после» заметно ниже, чем в окне «до». Прежняя
гипотеза — «обрезанный край окна «после»» — отвергнута: окна просто разной длины.

Если считать, что покупка у партнёра приходит пуассоновским потоком с
интенсивностью λ в месяц, то доля клиентов хотя бы с одной покупкой за t месяцев
равна `1 − exp(−λt)`. Из доли за 7 месяцев получаем λ, из λ — предсказание на
3 месяца, и сравниваем с фактом. Порог объявляется до прогона: расхождение до
3 п.п. считаем согласием, больше — содержательным отклонением, которое надо
назвать, а не списать на шум.


In [20]:
WINDOW_BEFORE_MONTHS = 7.0   # «с 2022-01-01 по 2022-08-01»
WINDOW_AFTER_MONTHS = 3.0    # «последние 3 месяца», точные границы неизвестны
TOLERANCE_PP = 3.0           # порог согласия, процентные пункты

rows = []
for before, after, label in [
    ("cb_bank_before", "cb_bank_last_3_month", "банковский кэшбэк"),
    ("cb_merch_before", "cb_merch_last_3_month", "партнёрский кэшбэк"),
]:
    share_long = (df[before] > 0).mean()
    share_short = (df[after] > 0).mean()
    lam = -np.log(1 - share_long) / WINDOW_BEFORE_MONTHS
    predicted = 1 - np.exp(-lam * WINDOW_AFTER_MONTHS)
    rows.append({
        "величина": label,
        "доля>0 за 7 мес": share_long,
        "λ в месяц": lam,
        "предсказание на 3 мес": predicted,
        "факт за 3 мес": share_short,
        "расхождение, п.п.": (share_short - predicted) * 100,
    })
windows = pd.DataFrame(rows)
print(windows.round(4).to_string(index=False))

for _, row in windows.iterrows():
    gap = abs(row["расхождение, п.п."])
    verdict = "сошлось" if gap <= TOLERANCE_PP else "разошлось, объяснимо"
    print(f"\n{row['величина']}: расхождение {row['расхождение, п.п.']:+.1f} п.п. -> {verdict}")
print("\nвердикт проверки 22: длина окна объясняет падение доли по банковскому кэшбэку;")
print("по партнёрскому покупок в окне «после» больше экстраполяции — это содержательное")
print("отклонение, согласуется с работающей программой, но сезонность не исключена")

          величина  доля>0 за 7 мес  λ в месяц  предсказание на 3 мес  факт за 3 мес  расхождение, п.п.
 банковский кэшбэк           0.6569     0.1528                 0.3677         0.3838             1.6050
партнёрский кэшбэк           0.6737     0.1600                 0.3812         0.4795             9.8352

банковский кэшбэк: расхождение +1.6 п.п. -> сошлось

партнёрский кэшбэк: расхождение +9.8 п.п. -> разошлось, объяснимо

вердикт проверки 22: длина окна объясняет падение доли по банковскому кэшбэку;
по партнёрскому покупок в окне «после» больше экстраполяции — это содержательное
отклонение, согласуется с работающей программой, но сезонность не исключена


### Суммы на месяц — дефект, который объяснить нечем

Доли объясняются длиной окна. Суммы — нет. Ниже медианы среди клиентов с
ненулевым кэшбэком, приведённые к одному месяцу. Если окна действительно 7 и 3
месяца, суммы на месяц должны быть сопоставимы; если они расходятся вдвое, то
либо программа выросла к окну «после», либо длина окна «до» в документации
указана неверно.


In [21]:
rows = []
for before, after, label in [
    ("cb_merch_before", "cb_merch_last_3_month", "партнёрский"),
    ("cb_bank_before", "cb_bank_last_3_month", "банковский"),
]:
    median_long = df.loc[df[before] > 0, before].median()
    median_short = df.loc[df[after] > 0, after].median()
    rows.append({
        "кэшбэк": label,
        "медиана за 7 мес, ₽": median_long,
        "₽/мес (÷7)": median_long / WINDOW_BEFORE_MONTHS,
        "медиана за 3 мес, ₽": median_short,
        "₽/мес (÷3)": median_short / WINDOW_AFTER_MONTHS,
        "во сколько раз выше": (median_short / WINDOW_AFTER_MONTHS)
        / (median_long / WINDOW_BEFORE_MONTHS),
        "₽/мес, если «до» = 3 мес": median_long / WINDOW_AFTER_MONTHS,
    })
amounts = pd.DataFrame(rows)
print(amounts.round(1).to_string(index=False))
print("\nвердикт: НЕ ВЫПОЛНЕНА — различить рост программы и неточность границ окна")
print("по самим данным нельзя. Оба типа кэшбэка на месяц выше примерно вдвое; при")
print("окне «до» длиной 3 месяца суммы на месяц сошлись бы почти точно (последняя")
print("колонка против «₽/мес (÷3)»). Записано в открытые вопросы паспорта.")
print("\nДля работы это не критично: из выгрузки берутся форма распределений и уровни")
print("отклика, а не абсолютные суммы. Но экономика главы 3 считается на окне «после»,")
print("где длина заявлена явно.")

     кэшбэк  медиана за 7 мес, ₽  ₽/мес (÷7)  медиана за 3 мес, ₽  ₽/мес (÷3)  во сколько раз выше  ₽/мес, если «до» = 3 мес
партнёрский                216.0        30.9                206.0        68.7                  2.2                      72.0
 банковский                128.0        18.3                120.0        40.0                  2.2                      42.7

вердикт: НЕ ВЫПОЛНЕНА — различить рост программы и неточность границ окна
по самим данным нельзя. Оба типа кэшбэка на месяц выше примерно вдвое; при
окне «до» длиной 3 месяца суммы на месяц сошлись бы почти точно (последняя
колонка против «₽/мес (÷3)»). Записано в открытые вопросы паспорта.

Для работы это не критично: из выгрузки берутся форма распределений и уровни
отклика, а не абсолютные суммы. Но экономика главы 3 считается на окне «после»,
где длина заявлена явно.


## Наблюдательная разница — и почему она ничего не значит

In [22]:
got = df["cnt_communication"] > 0
print(f"с коммуникацией:  n={got.sum():6d}  доля целевого действия {y[got].mean():.4f}")
print(f"без коммуникации: n={(~got).sum():6d}  доля целевого действия {y[~got].mean():.4f}")
print(f"разница: {(y[got].mean() - y[~got].mean()) * 100:.1f} п.п.")
print("\nЭто НЕ оценка эффекта: правило отбора получателей неизвестно.")
print("Число идёт в главу 1 как иллюстрация, почему нужен эксперимент.")


с коммуникацией:  n= 32858  доля целевого действия 0.5322
без коммуникации: n= 17142  доля целевого действия 0.3785
разница: 15.4 п.п.

Это НЕ оценка эффекта: правило отбора получателей неизвестно.
Число идёт в главу 1 как иллюстрация, почему нужен эксперимент.


## MDE по дизайнам

`d = (z_{1−α/2} + z_{power}) · sqrt(p(1−p)·(1/n_t + 1/n_c))`

Базовая доля `p` берётся из окна «после», то есть это **трёхмесячная** величина.
Все MDE ниже — тоже про трёхмесячную кампанию; переносить их на месяц или на год
нельзя, длина окна в такой пересчёт не входит линейно.


In [23]:
def mde_abs(p: float, n_treat: int, n_control: int, z_sum: float = Z_SUM) -> float:
    """Минимальный ловимый эффект, абсолютная разность долей."""
    return z_sum * np.sqrt(p * (1 - p) * (1 / n_treat + 1 / n_control))


p_base = y.mean()
n_total = len(df)
rows = []
for control_share in (0.5, 0.3, 0.1):
    n_c = int(n_total * control_share)
    n_t = n_total - n_c
    d = mde_abs(p_base, n_t, n_c)
    rows.append({"доля контроля": control_share, "n_контроль": n_c,
                 "MDE абс., п.п.": round(d * 100, 2),
                 "MDE отн., %": round(d / p_base * 100, 1)})
mde_table = pd.DataFrame(rows)
print(f"базовая доля p = {p_base:.4f}, всего клиентов {n_total}")
print(mde_table.to_string(index=False))

n_needed = (Z_SUM ** 2) * 2 * p_base * (1 - p_base) / (0.01 ** 2)
print(f"\nдля MDE = 1 п.п. нужно {n_needed:,.0f} клиентов в группе — на выборке недостижимо")


базовая доля p = 0.4795, всего клиентов 50000
 доля контроля  n_контроль  MDE абс., п.п.  MDE отн., %
           0.5       25000            1.25          2.6
           0.3       15000            1.37          2.8
           0.1        5000            2.09          4.4

для MDE = 1 п.п. нужно 39,179 клиентов в группе — на выборке недостижимо
